# Aula 06/04

Conjunto de teste e treino.

O modelo deve prever a próxima palavra.

# Requisitos do trabalho
O sistema deve:

1. Baixar e preparar os dados do corpus Gutenberg usando NLTK.

2. Pré-processar o texto, incluindo ao menos:

* tokenização;

* normalização básica, como conversão para minúsculas;

* tratamento de pontuação, conforme a estratégia adotada.

3. Treinar offline um modelo de linguagem para predição da próxima palavra.

4. Salvar o modelo treinado em arquivo, para que ele não precise ser refeito a cada execução.

5. Carregar o modelo em memória principal no momento da execução do programa.

6. Receber como entrada uma sequência de palavras e retornar:

* a palavra sugerida como continuação mais provável;

* uma lista das palavras mais prováveis, acompanhadas de suas probabilidades.

7. Avaliar o modelo por meio da perplexidade, calculada sobre um conjunto de teste separado do conjunto de treinamento.

8. Aplicar algum método de suavização para lidar com eventos raros e palavras fora do vocabulário (OOV – Out Of Vocabulary).

# Tratamento de OOV e suavização
O trabalho deve incluir uma estratégia explícita para palavras não vistas no treinamento. Algumas possibilidades aceitas são:

* substituição de palavras raras por um token especial, como <UNK>;

* suavização de Laplace (add-one);

* suavização de Lidstone;

* Good-Turing;

* Kneser-Ney;

* outro método equivalente, desde que devidamente justificado.

O aluno deve explicar qual técnica foi adotada e por que ela foi escolhida.

# Treinamento e avaliação
O treinamento deve ser realizado offline, e o modelo resultante deve ser armazenado em arquivo. Na execução normal do sistema, o programa deve apenas carregar esse arquivo e realizar as predições.

Além da geração das sugestões, o trabalho deve apresentar uma avaliação quantitativa do modelo, incluindo:

* divisão entre conjunto de treino e conjunto de teste;

* cálculo da perplexidade no conjunto de teste;

* breve análise dos resultados obtidos.

In [ ]:
import nltk
nltk.download('gutenberg')

from nltk.corpus import gutenberg
print(gutenberg.fileids())

['austen-emma.txt', 'austen-persuasion.txt', 'austen-sense.txt', 'bible-kjv.txt', 'blake-poems.txt', 'bryant-stories.txt', 'burgess-busterbrown.txt', 'carroll-alice.txt', 'chesterton-ball.txt', 'chesterton-brown.txt', 'chesterton-thursday.txt', 'edgeworth-parents.txt', 'melville-moby_dick.txt', 'milton-paradise.txt', 'shakespeare-caesar.txt', 'shakespeare-hamlet.txt', 'shakespeare-macbeth.txt', 'whitman-leaves.txt']


[nltk_data] Downloading package gutenberg to /root/nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!


# Carregando o texto selecionado

# Tokenizando e normalizando

* tokenização;

* normalização básica, como conversão para minúsculas;

* tratamento de pontuação, conforme a estratégia adotada.

# Prevendo a probabilidade da palavra

Tratar palavra do fora do vocabulário -> suavizar

# Trabalho

In [1]:
import nltk
from nltk.corpus import gutenberg
import re
from collections import Counter
import math
import joblib
import os

# Baixa os pacotes do NLTK no Colab
nltk.download('gutenberg', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

Bibliotecas importadas com sucesso!


In [2]:
def load_file(file_id='chesterton-brown.txt'):
    sentencas = gutenberg.sents(file_id)
    return sentencas

def normalizacao_texto(sentencas):
    sentencas_normalizadas = []
    for sentenca in sentencas:
        sentenca_normal = []
        for palavra in sentenca:
            palavra_minuscula = palavra.lower()
            limpeza_palavra = re.sub(r'[^\w\s]', '', palavra_minuscula)
            if limpeza_palavra != '':
                sentenca_normal.append(limpeza_palavra)
        if len(sentenca_normal) > 0:
            sentencas_normalizadas.append(sentenca_normal)
    return sentencas_normalizadas

def divisao_teste_treino(sentencas, proporcao_treino=0.8):
    divisao = int(len(sentencas) * proporcao_treino)
    treino = sentencas[:divisao]
    teste = sentencas[divisao:]
    return treino, teste

def tratar_ovv(treino, teste, frequencia_minima=2):
    contagem_palavras = Counter()
    for sentenca in treino:
        contagem_palavras.update(sentenca)

    vocabulario_permitido = set()
    for palavra, contagem in contagem_palavras.items():
        if contagem >= frequencia_minima:
            vocabulario_permitido.add(palavra)

    vocabulario_permitido.add('<UNK>')

    def aplicar_unk(sentencas, vocabulario):
        sentencas_processadas = []
        for sentenca in sentencas:
            nova_sentenca = []
            for palavra in sentenca:
                if palavra in vocabulario:
                    nova_sentenca.append(palavra)
                else:
                    nova_sentenca.append('<UNK>')
            sentencas_processadas.append(nova_sentenca)
        return sentencas_processadas

    treino_unk = aplicar_unk(treino, vocabulario_permitido)
    teste_unk = aplicar_unk(teste, vocabulario_permitido)

    return treino_unk, teste_unk, vocabulario_permitido

print("Funções de pré-processamento carregadas!")

Funções de pré-processamento carregadas!


In [3]:
class ModeloLinguagem:
    def __init__(self, alfa=0.1):
        self.alfa = alfa
        self.trigramas = Counter()
        self.bigramas = Counter()
        self.palavras_permitidas = set()
        self.tamanho_vocabulario = 0

    def treinar(self, sentencas_treino, vocabulario_do_preprocess):
        self.vocabulario_permitido = vocabulario_do_preprocess
        self.tamanho_vocabulario = len(self.vocabulario_permitido)

        for sentenca in sentencas_treino:
            for i in range(2, len(sentenca)):
                palavra_atual = sentenca[i]
                palavra_anterior = sentenca[i-1]
                palavra_retrasada = sentenca[i-2]

                tupla_contexto = (palavra_retrasada, palavra_anterior)
                self.bigramas[tupla_contexto] += 1

                tupla_trigrama = (palavra_retrasada, palavra_anterior, palavra_atual)
                self.trigramas[tupla_trigrama] += 1

    def calcular_probabilidade(self, palavra_retrasada, palavra_anterior, palavra_atual):
        tupla_trigrama = (palavra_retrasada, palavra_anterior, palavra_atual)
        tupla_contexto = (palavra_retrasada, palavra_anterior)

        contagem_trigrama = self.trigramas[tupla_trigrama]
        contagem_contexto = self.bigramas[tupla_contexto]

        probabilidade_palavra = (contagem_trigrama + self.alfa) / (contagem_contexto + (self.alfa * self.tamanho_vocabulario))
        return probabilidade_palavra

    def sugerir_palavra(self, texto_input, top_n=3):
        tokens = texto_input.lower().split()
        if len(tokens) < 2:
            return "Digite pelo menos duas palavras para eu prever a próxima!", []

        ultima_palavra = tokens[-1]
        penultima_palavra = tokens[-2]

        if ultima_palavra not in self.vocabulario_permitido:
            ultima_palavra = '<UNK>'
        if penultima_palavra not in self.vocabulario_permitido:
            penultima_palavra = '<UNK>'

        palavras_sugeridas = []
        for palavra_candidata in self.vocabulario_permitido:
            probabilidade = self.calcular_probabilidade(penultima_palavra, ultima_palavra, palavra_candidata)
            palavras_sugeridas.append((probabilidade, palavra_candidata))

        palavras_sugeridas.sort(reverse=True)
        podio = palavras_sugeridas[:top_n]
        palavra_vencedora = podio[0][1]

        return palavra_vencedora, podio

print("Classe do Modelo carregada!")

Classe do Modelo carregada!


In [4]:
print("1. Carregando e limpando texto...")
texto_bruto = load_file()
texto_limpo = normalizacao_texto(texto_bruto)

print("2. Dividindo Treino e Teste...")
treino, teste = divisao_teste_treino(texto_limpo)

print("3. Tratando OOV...")
treino_final, teste_final, vocabulario = tratar_ovv(treino, teste)

print("4. Treinando o Modelo de Trigramas...")
modelo = ModeloLinguagem(alfa=0.1)
modelo.treinar(treino_final, vocabulario)

print("5. Salvando em disco (ambiente Colab)...")
os.makedirs('models', exist_ok=True)
joblib.dump(modelo, 'models/meu_modelo_n_grama.pkl')
joblib.dump(teste_final, 'models/dados_teste.pkl')
print("Sucesso! Modelo salvo em 'models/'")

1. Carregando e limpando texto...
2. Dividindo Treino e Teste...
3. Tratando OOV...
4. Treinando o Modelo de Trigramas...
5. Salvando em disco (ambiente Colab)...
Sucesso! Modelo salvo em 'models/'


In [5]:
def calcular_perplexidade():
    try:
        modelo_carregado = joblib.load('models/meu_modelo_n_grama.pkl')
        dados_teste_carregados = joblib.load('models/dados_teste.pkl')
    except FileNotFoundError:
        print("Arquivos não encontrados.")
        return None

    N = 0
    denominador = 0

    for sentenca in dados_teste_carregados:
        for i in range(2, len(sentenca)): # Adaptado para trigramas
            palavra_atual = sentenca[i]
            palavra_anterior = sentenca[i-1]
            palavra_retrasada = sentenca[i-2]

            probabilidade = modelo_carregado.calcular_probabilidade(palavra_retrasada, palavra_anterior, palavra_atual)
            denominador += math.log2(probabilidade)
            N += 1

    expoente = -(denominador / N)
    perplexidade = math.pow(2, expoente)
    return perplexidade

resultado = calcular_perplexidade()
if resultado is not None:
    print(f"Perplexidade do modelo de Trigramas: {resultado:.4f}")

Perplexidade do modelo de Trigramas: 1502.4404


In [18]:
# Carrega o modelo para uso
meu_modelo = joblib.load('models/meu_modelo_n_grama.pkl')

frase_teste = input("Digite pelo menos duas palavras em inglês (ex: 'father brown'): ")

vencedora, podio = meu_modelo.sugerir_palavra(frase_teste, top_n=5)

print(f"\n-> A continuação mais provável para '{frase_teste}' é: '{vencedora}'\n")
print("Top 5 Probabilidades:")
for prob, pal in podio:
    print(f"{pal}: {prob:.6f}")

Digite pelo menos duas palavras em inglês (ex: 'father brown'): the church

-> A continuação mais provável para 'the church' é: 'excuse'

Top 5 Probabilidades:
excuse: 0.003217
youth: 0.000292
yourselves: 0.000292
yourself: 0.000292
yours: 0.000292
